In [ ]:
import os
import json
import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA

In [51]:
MODEL_NAME = "Captain-1337/CrudeBERT"
BASE_TOKENIZER = "bert-base-uncased"

print("모델 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(BASE_TOKENIZER)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

모델 로딩 중...


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [52]:
input_file = "data/final_treated_brent_oil_articles_analyzed_updated.json"
output_file = "data/final_treated_brent_oil_articles_embedded2.json"

print(f"입력 파일 로드 중: {input_file}")
with open(input_file, "r", encoding="utf-8") as f:
    articles = json.load(f)

print(f"총 {len(articles)}개의 기사 로드 완료\n")

입력 파일 로드 중: data/final_treated_brent_oil_articles_analyzed_updated.json
총 6080개의 기사 로드 완료



In [ ]:
def get_embedding(text, max_length=512):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    )
    
    with torch.no_grad():
        outputs = model(**inputs)

        embedding = outputs.last_hidden_state[:, 0, :].numpy()
    
    return embedding[0]

def get_decoding(vector):
    with torch.no_grad():
        vector_tensor = torch.tensor(vector).unsqueeze(0)
        decoded = model.decode(vector_tensor)
    return decoded

In [54]:
target_dimension = 64

In [55]:
print(f"총 {len(articles)}개 article의 summary를 임베딩 중...")
embeddings_list = []
valid_indices = []

총 6080개 article의 summary를 임베딩 중...


In [56]:
for i, article in enumerate(tqdm(articles, desc="임베딩 생성")):
    if 'summary' in article and article['summary']:
        try:
            embedding = get_embedding(article['summary'])
            embeddings_list.append(embedding)
            valid_indices.append(i)
        except Exception as e:
            print(f"경고: article {i} 임베딩 생성 실패: {e}")
            article['summary_embedding'] = None
    else:
        print(f"경고: article {i}에 summary가 없습니다.")
        article['summary_embedding'] = None

임베딩 생성:  41%|████      | 2466/6080 [12:46<07:33,  7.98it/s] 

경고: article 2462에 summary가 없습니다.
경고: article 2463에 summary가 없습니다.
경고: article 2464에 summary가 없습니다.


임베딩 생성:  56%|█████▌    | 3398/6080 [17:32<15:04,  2.96it/s]

경고: article 3398에 summary가 없습니다.


임베딩 생성:  56%|█████▋    | 3420/6080 [17:39<09:35,  4.62it/s]

...

임베딩 생성:  96%|█████████▌| 5821/6080 [29:05<01:12,  3.58it/s]

경고: article 5821에 summary가 없습니다.


임베딩 생성:  96%|█████████▌| 5830/6080 [29:07<00:52,  4.80it/s]

경고: article 5830에 summary가 없습니다.


임베딩 생성: 100%|██████████| 6080/6080 [30:15<00:00,  3.35it/s]


In [57]:
embeddings_array = np.array(embeddings_list)
n_samples = len(embeddings_array)

if n_samples < target_dimension:
    actual_dimension = n_samples
    print(f"\n경고: 샘플 수({n_samples})가 목표 차원({target_dimension})보다 적습니다.")
    print(f"차원 축소 생략: 768차원 그대로 사용")
    reduced_embeddings = embeddings_array
else:
    print(f"\n차원 축소 중: 768 -> {target_dimension}")
    pca = PCA(n_components=target_dimension)
    reduced_embeddings = pca.fit_transform(embeddings_array)
    print(f"설명된 분산 비율: {pca.explained_variance_ratio_.sum():.4f}")

for idx, article_idx in enumerate(valid_indices):
    articles[article_idx]['summary_embedding'] = reduced_embeddings[idx].tolist()

os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(articles, f, ensure_ascii=False, indent=2)

print(f"\n결과 저장 완료: {output_file}")
print(f"총 {len(valid_indices)}개의 임베딩 생성됨")


차원 축소 중: 768 -> 64
설명된 분산 비율: 0.8500

결과 저장 완료: data/final_treated_brent_oil_articles_embedded2.json
총 6071개의 임베딩 생성됨
